# Reto 05 — Pandas II: cruzar, agregar y reportar

**Caso:** tienes tres tablas sueltas —ventas, clientes y productos— y una reunion en dos horas.
Te piden una tabla unica lista para reportar y 4 respuestas.

El tema real de este reto no es `merge`: es **no perder filas sin darte cuenta**.
Un `merge` mal elegido borra plata en silencio y nadie te avisa.

In [ ]:
import pandas as pd

DS = "../../_RECURSOS/datasets"

ventas    = pd.read_csv(f"{DS}/ventas.csv",    sep=";", decimal=",", encoding="utf-8")
clientes  = pd.read_csv(f"{DS}/clientes.csv",  sep=";", encoding="utf-8")
productos = pd.read_csv(f"{DS}/productos.csv", sep=";", encoding="utf-8")

print(ventas.shape, clientes.shape, productos.shape)

## 1 — Limpiar ventas

Reusa lo del reto 04: deduplicar, normalizar `canal`, tipar `fecha` (`dayfirst=True`),
recalcular importes nulos. Guarda en `ventas`.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
assert len(ventas) == 482
assert ventas["fecha"].dtype.kind == "M"
assert ventas["importe"].isna().sum() == 0
print("OK 1 —", ventas.shape)

## 2 — El diagnostico ANTES del merge

No cruces nada todavia. Primero averigua:

1. Cuantos `id_cliente` de `ventas` **no existen** en `clientes`?
2. Cuantos clientes **no tienen ninguna venta**?
3. Todos los `id_producto` de ventas existen en `productos`?

Pista: `~ventas["id_cliente"].isin(clientes["id_cliente"])`.

Este paso es el que separa a alguien que sabe pandas de alguien que sabe analisis de datos.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
huerfanas = ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"]), "id_cliente"].unique()
sin_ventas = clientes.loc[~clientes["id_cliente"].isin(ventas["id_cliente"]), "id_cliente"].unique()
assert len(huerfanas) == 1 and huerfanas[0] == "C777", f"Salio {huerfanas}"
assert len(sin_ventas) == 1 and sin_ventas[0] == "C099", f"Salio {sin_ventas}"
print("OK 2 — venta con cliente inexistente:", huerfanas, "| cliente sin ventas:", sin_ventas)

### Tus hallazgos

*(que significa cada uno de esos dos casos para el negocio, no para pandas)*

## 3 — `inner` vs `left`: pruebalo

Haz **los dos** merges de `ventas` con `clientes` y compara cuantas filas quedan.

In [ ]:
con_inner = ...
con_left  = ...


# --- verificacion (no modificar) ---
assert len(con_inner) == 481, f"inner deberia dar 481, salio {len(con_inner)}"
assert len(con_left) == 482, f"left deberia dar 482, salio {len(con_left)}"
print(f"OK 3 — inner: {len(con_inner)} | left: {len(con_left)} | se pierde 1 fila")

### Cual eliges y por que

*(la fila que pierde el `inner` vale plata real. Que decides?)*

## 4 — La tabla maestra

Construye `maestra`: ventas + datos del cliente + datos del producto.

Requisitos:
- No se puede perder ninguna venta
- Sufijos claros si hay nombres de columna repetidos
- Usa `validate=` en cada merge para que pandas te avise si la relacion no es la que crees

In [ ]:
maestra = ...


# --- verificacion (no modificar) ---
assert len(maestra) == 482, f"Se perdieron filas: {len(maestra)}"
for col in ("nombre", "segmento", "producto", "categoria"):
    assert col in maestra.columns, f"falta la columna {col}"
print("OK 4 —", maestra.shape)

## 5 — Agregaciones

Resuelve con `groupby` + `agg`:

1. Por `segmento`: importe total, numero de ventas y ticket promedio, **en una sola llamada**
2. Por `categoria` y `canal`: importe total
3. El mes con mayor facturacion de cada ciudad

In [ ]:
# TODO 5.1

In [ ]:
# TODO 5.2

In [ ]:
# TODO 5.3

## 6 — Pivot

Arma una `pivot_table` con ciudades en filas, meses en columnas e importe total en las celdas.
Agrega totales de fila y columna (`margins=True`).

In [ ]:
# TODO


# --- verificacion (no modificar) ---
print("OK 6")

## 7 — Exportar

Guarda `maestra` como `ventas_maestra.csv` (`sep=";"`, `decimal=","`, `index=False`).

In [ ]:
# TODO


# --- verificacion (no modificar) ---
import os
assert os.path.exists("ventas_maestra.csv")
print("OK 7")

## 8 (opcional, +2 puntos) — API

Trae datos de una API publica y cruzalos. Sugerencia: tipo de cambio de
`https://open.er-api.com/v6/latest/USD` para expresar la facturacion en dolares.

Si no hay internet, di explicitamente que no pudiste y muestra el `try/except` que lo maneja.

In [ ]:
# TODO

---
## Criterios de aceptacion

- [ ] *Restart & Run All* corre limpio y todos los `OK` salen
- [ ] El diagnostico del punto 2 esta hecho **antes** del primer merge
- [ ] `maestra` tiene 482 filas: no se perdio ninguna venta
- [ ] Cada `merge` lleva `how=` explicito y `validate=`
- [ ] Las dos celdas de decision (punto 2 y punto 3) estan escritas
- [ ] El punto 5.1 usa **una sola** llamada a `agg`, no tres `groupby` separados
- [ ] Se genera `ventas_maestra.csv`